### Generating Polynomials and Divisibility Conditions

In Section 2 of the manuscript, Faulhaber's formula is utilized to express the sum of consecutive powers $S_k(x)$ as a polynomial in $x$ with rational coefficients. We factor this polynomial as:
$$S_k(x) = \frac{1}{C_k} x(x+1)(2x+1)T_k(x) \quad \text{(for even } k\text{)},$$
$$S_k(x) = \frac{1}{C_k} x^2(x+1)^2 T_k(x) \quad \text{(for odd } k\text{)}.$$

The script below generates $S_k(x)$ for any exponent $k$, extracts the constant $C_k$, and isolates the polynomial $T_k(x)$. It then calculates the possible greatest common divisors between $T_k(x)$ and the surrounding linear factors, generating the data found in **Table 1** of the manuscript. 

Furthermore, for even values of $k \geq 4$, it computes the polynomials $a(x)$ and $b(x)$ and the integer $r$ that satisfy the identity $a(x)T_k(x) + b(x)(2x+1) = r$, generating the data found in **Table 2**.

In [ ]:
# =============================================================================
# Faulhaber Polynomial Generation and Table Data Extraction
# =============================================================================

# Declare symbolic variables for the polynomial ring
var('u', 'v', 'x')

def S(k):
    """
    Returns S_k(x) as a polynomial in x using Faulhaber's formula.
    """
    return (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )

def generate_table1_data(k):
    """
    Generates the polynomial T_k(x) and its divisibility conditions.
    Returns: [k, C_k, T_k(x), T_k(u), gcd(u, T_k), gcd(C_k, T_k)]
    """
    Sk = factor(S(k))
    evenfac = x * (x + 1) * (2 * x + 1)
    oddfac = x^2 * (x + 1)^2
    
    Skcoefs = Sk.coefficients()
    Ck = lcm(coef[0].denominator() for coef in Skcoefs)
    
    # Substitution variable u = x(x+1)
    usubs = (-1 + sqrt(1 + 4 * u)) / 2
    
    if k == 1:
        return None
    elif k % 2 == 0:
        Tk = factor(simplify(Ck * Sk / evenfac))
    else:
        Tk = factor(simplify(Ck * Sk / oddfac))
        
    Tku = expand(Tk(x = usubs))
    Tkcoefs = Tku.coefficients(sparse=False)
    
    # Determine gcd(u, T_k(x))
    first_coef = Integer(Tkcoefs[0])
    if first_coef % 2 == 0:
        gcduTk = {d for d in divisors(first_coef) if d % 2 == 0}
    else:
        gcduTk = set(divisors(first_coef))
        
    # Determine gcd(C_k, T_k(x))
    gcdCkTk = {gcd(Tk(x=i), Ck) for i in range(Ck)}
    
    return [k, Ck, Tk, Tku, gcduTk, gcdCkTk]

def generate_table2_data(k):
    """
    Generates the coefficients mapping T_k(x) to v = 2x+1.
    Finds a(x), b(x), and r satisfying a(x)T_k(x) + b(x)v = r.
    Returns: [k, a(x), b(x), r]
    """
    Tk = generate_table1_data(k)[2]
    
    # Substitution variable v = 2x+1
    Tkv = expand(Tk(x = (v - 1) / 2))
    Tkvcoefs = Tkv.coefficients()
    Tkvcoefs2 = Tkv.coefficients(sparse=False)
    
    aval = lcm(coef[0].denominator() for coef in Tkvcoefs)
    rval = aval * Tkvcoefs2[0]
    signr = sgn(rval)
    bval = expand((aval * Tkv - rval) / v)
    
    return [k, signr * aval, bval, Integer(signr * rval)]

# =============================================================================
#  Displaying the Tables
# =============================================================================

print("=" * 90)
print("TABLE 1: C_k and Divisibility Data for T_k(x) [2 <= k <= 11]")
print("=" * 90)
print(f"{'k':<4} | {'C_k':<4} | {'gcd(u, T_k)':<15} | {'gcd(C_k, T_k)':<15} | {'T_k(x)':<30}")
print("-" * 90)

for k in range(2, 12):
    data = generate_table1_data(k)
    if data:
        k_val, Ck, Tk, Tku, gcdu, gcdCk = data
        # Cast all SageMath objects to Python strings before formatting
        print(f"{str(k_val):<4} | {str(Ck):<4} | {str(gcdu):<15} | {str(gcdCk):<15} | {str(Tk):<30}")

print("\n" + "=" * 80)
print("TABLE 2: a(x)T_k(x) + b(x)v = r  [Even k >= 4]")
print("=" * 80)
print(f"{'k':<4} | {'a(x)':<6} | {'r':<8} | {'b(x)':<30}")
print("-" * 80)

for k in range(4, 11, 2):
    data = generate_table2_data(k)
    k_val, a, b, r = data
    # Cast all SageMath objects to Python strings before formatting
    print(f"{str(k_val):<4} | {str(a):<6} | {str(r):<8} | {str(b):<30}")

TABLE 1: C_k and Divisibility Data for T_k(x) [2 <= k <= 11]
k    | C_k  | gcd(u, T_k)     | gcd(C_k, T_k)   | T_k(x)                        
------------------------------------------------------------------------------------------
2    | 6    | {1}             | {1}             | 1                             
3    | 4    | {1}             | {1}             | 1                             
4    | 30   | {1}             | {1, 5}          | 3*x^2 + 3*x - 1               
5    | 12   | {1}             | {1, 3}          | 2*x^2 + 2*x - 1               
6    | 42   | {1}             | {1, 7}          | 3*x^4 + 6*x^3 - 3*x + 1       
7    | 24   | {2}             | {2, 6}          | 3*x^4 + 6*x^3 - x^2 - 4*x + 2 
8    | 90   | {1, 3}          | {3, 15}         | 5*x^6 + 15*x^5 + 5*x^4 - 15*x^3 - x^2 + 9*x - 3
9    | 20   | {1, 3}          | {1, 5}          | (2*x^4 + 4*x^3 - x^2 - 3*x + 3)*(x^2 + x - 1)
10   | 66   | {1, 5}          | {1, 11}         | (3*x^6 + 9*x^5 + 2*x^4 - 11*x^3 + 3*x